In [ ]:
#Single server queue - 1
import simpy, random

def customer(env, name, server):
    t = env.now
    print(name, "arrives at", t)

    with server.request() as req:
        yield req
        print(name, "waits", env.now - t)

        yield env.timeout(random.expovariate(1))
        print(name, "leaves at", env.now)

def setup(env):
    server = simpy.Resource(env, capacity=1)

    for i in range(5):
        env.process(customer(env, f"Cust {i}", server))
        yield env.timeout(random.expovariate(0.5))

env = simpy.Environment()
env.process(setup(env))
env.run(until=20)

#print("Average waiting time:", round(sum(wait)/len(wait), 2))

In [ ]:
#Inventory management - 2
import random
from collections import deque

class Inventory:
    def __init__(self):
        self.stock = deque()
        self.waste = 0

    def add(self):
        self.stock.append([10, 3])

    def sell(self, n):
        while n > 0 and self.stock:
            q, d = self.stock.popleft()
            sold = min(q, n)
            n -= sold
            if q > sold:
                self.stock.appendleft([q-sold, d])

    def update(self):
        temp = deque()
        for q, d in self.stock:
            d -= 1
            if d == 0:
                self.waste += q
            else:
                temp.append([q, d])
        self.stock = temp

    def report(self, day):
        print("Day:", day)
        print("Stock:", sum(q for q, d in self.stock))
        print("Waste:", self.waste)

item = Inventory()

for day in range(1, 6):
    item.add()
    item.sell(random.randint(8, 12))
    item.update()
    item.report(day)

In [ ]:
#Discrete Vs Continuous  - 3
#Newtons law of cooling
import numpy as np
import matplotlib.pyplot as plt

T0, Tr, k, dt = 100, 25, 0.1, 1
t = np.arange(0, 50, dt)

# Continuous model
Tc = Tr + (T0-Tr) * np.exp(-k*t)

# Discrete model (Euler)
Td = [T0]
for i in range(len(t)-1):
    Td.append(Td[-1] - k*(Td[-1]-Tr)*dt)

# Plot
plt.plot(t, Tc, label="Continuous")
plt.plot(t, Td, 'o', label="Discrete")
plt.xlabel("Time")
plt.ylabel("Temperature")
plt.legend()
plt.grid()
plt.show()

In [ ]:
#Components of a system - 4
#vending machine
def vending():
    items = {"Chips": 20, "Soda": 30, "Chocolate": 25}

    print(items)
    choice = input("Select item: ")
    money = int(input("Enter money: "))

    if choice not in items:
        print("Invalid selection")
    elif money < items[choice]:
        print("Not enough money")
    else:
        print("Dispensed:", choice)
        print("Change: ₹", money - items[choice])

vending()

In [ ]:
#Poisson Process - 5
#compare simulation with theoretical poisson distribution
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson

lam = 5
a = np.random.poisson(lam, 1000)

x, c = np.unique(a, return_counts=True)

plt.bar(x, c/1000, label="Simulation")
plt.plot(x, poisson.pmf(x, lam), 'ro-', label="Theoretical")
plt.xlabel("Arrivals")
plt.ylabel("Probability")
plt.legend()
plt.show()

In [ ]:
#Continuous distribution-normal & exponential - 6
#Simulating Exam Scores (Normal Distribution)
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

mu, sigma = 70, 10
data = np.random.normal(mu, sigma, 1000)

plt.hist(data, bins=30, density=True, alpha=0.6)

x = np.linspace(30, 110, 100)
plt.plot(x, norm.pdf(x, mu, sigma), 'r')

plt.xlabel("Score")
plt.ylabel("Density")
plt.title("Normal Distribution")
plt.show()

print("Mean:", np.mean(data))
print("Standard Deviation:", np.std(data))

#Taxi Waiting Times (Exponential Distribution)
import numpy as np
import matplotlib.pyplot as plt

mean = 10
lam = 1 / mean
data = np.random.exponential(mean, 1000)

plt.hist(data, bins=30, density=True, alpha=0.6)

x = np.linspace(0, max(data), 100)
plt.plot(x, lam * np.exp(-lam*x), 'r')

plt.title("Exponential Distribution")
plt.xlabel("Waiting Time (minutes)")
plt.ylabel("Density")
plt.show()

print("Sample mean:", np.mean(data))
print("Theoretical mean:", mean)

In [ ]:
#M/M/1 & M/M/C - 7
#Problem 1 & 2: Single Server Queue Simulation (M/M/1 Model)
import numpy as np

lam, mu = 2, 3
N = 10000

arrival = np.cumsum(np.random.exponential(1/lam, N))
service = np.random.exponential(1/mu, N)

finish = 0
wait = []
system = []

for i in range(N):
    start = max(arrival[i], finish)
    finish = start + service[i]

    wait.append(start - arrival[i])
    system.append(finish - arrival[i])

print("Average waiting time (Wq):", np.mean(wait))
print("Average time in system (W):", np.mean(system))

#Multi-Server Call Center Simulation (M/M/c Model using Heap)
import numpy as np
import heapq

lam, mu = 10, 6
c, N = 2, 5000

arrival = np.cumsum(np.random.exponential(1/lam, N))
service = np.random.exponential(1/mu, N)

servers = [0] * c
wait = []
system = []

for i in range(N):
    available = heapq.heappop(servers)
    start = max(arrival[i], available)
    finish = start + service[i]

    wait.append(start - arrival[i])
    system.append(finish - arrival[i])

    heapq.heappush(servers, finish)

print("Average waiting time (Wq):", np.mean(wait))
print("Average time in system (W):", np.mean(system))

In [ ]:
#Test for pseudorandom numbers - 8
# Chi-square Goodness-of-Fit Test (Uniform Distribution - Fair Die)
import numpy as np
from scipy.stats import chi2

observed = np.array([8, 9, 10, 12, 11, 10])
expected = np.array([10] * 6)

chi = sum((observed - expected)**2 / expected)
df = 5
p = 1 - chi2.cdf(chi, df)

print("Chi-square:", chi)
print("Degrees of freedom:", df)
print("P-value:", p)

if p > 0.05:
    print("Die is fair")
else:
    print("Die is not fair")

In [ ]:
#Fit test - 9
#Fitting a Normal Distribution (Bank Waiting Times)
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, kstest

data = [7, 9, 8, 11, 13, 9, 12, 10, 14, 13,
        8, 9, 12, 10, 11, 13, 10, 9, 8, 12]

mean = np.mean(data)
std = np.std(data)

stat, p = kstest(data, 'norm', args=(mean, std))

print("Mean:", mean)
print("Standard Deviation:", std)
print("KS Statistic:", stat)
print("P-value:", p)

if p > 0.05:
    print("Data follows Normal Distribution")
else:
    print("Data does not follow Normal Distribution")

plt.hist(data, bins=8, density=True, alpha=0.6)
x = np.linspace(min(data), max(data), 100)
plt.plot(x, norm.pdf(x, mean, std), 'r')
plt.title("Normal Distribution")
plt.show()

#Fitting an Exponential Distribution (Ticket Service Times)
import numpy as np
from scipy.stats import expon, chi2

data = [2.1, 1.9, 3.5, 2.8, 1.7, 3.3, 2.5, 2.0, 3.1, 1.6,
        2.4, 2.9, 1.5, 3.0, 2.6, 2.2, 1.8, 3.4, 2.3, 2.7,
        3.2, 1.9, 2.8, 3.3, 2.1]

lam = 1 / np.mean(data)
bins = np.linspace(min(data), max(data), 6)

obs, _ = np.histogram(data, bins)
exp = len(data) * np.diff(expon.cdf(bins, scale=1/lam))

chi = np.sum((obs-exp)**2 / exp)
df = len(bins) - 2
p = 1 - chi2.cdf(chi, df)

print("Lambda:", round(lam, 4))
print("Chi-square:", round(chi, 4))
print("P-value:", round(p, 4))

if p > 0.05:
    print("Data follows Exponential Distribution")
else:
    print("Data does not follow Exponential Distribution")

In [ ]:
#Verification and validation - 10
#Single-Server Queue (M/M/1 Model) Verification & Validation
import random
import numpy as np

arrival, end, wait = [0], [], []

for i in range(1, 100):
    arrival.append(arrival[-1] + random.expovariate(1/5))

for i in range(100):
    start = max(arrival[i], end[-1] if end else 0)
    finish = start + random.expovariate(1/4)
    end.append(finish)
    wait.append(start - arrival[i])

lam, mu = 1/5, 1/4
Wq = lam / (mu * (mu - lam))

print("Verification:", "Passed" if all(end[i] >= arrival[i] for i in range(100)) else "Failed")
print("Simulation Waiting Time:", round(np.mean(wait), 2))
print("Theory Waiting Time:", round(Wq, 2))

if abs(np.mean(wait) - Wq) / Wq < 0.2:
    print("Validation Passed")
else:
    print("Validation Failed")